# Chương 4 (bài tập) – Hồi quy tuyến tính: Dự đoán giá nhà
*Giáo trình **Học máy – Cơ sở và Ứng dụng với Python*** · Notebook bài tập có **gợi ý từng bước**

## Cách làm bài
Mỗi bước gồm: **Yêu cầu → Gợi ý (bấm để mở từng cái một) → Ô code cần điền → Lời giải → Ô tự kiểm tra**.
1. Đọc yêu cầu và thử tự làm trước.
2. Kẹt thì mở **Gợi ý 1**, rồi Gợi ý 2… (mỗi gợi ý lộ thêm một chút).
3. Điền vào chỗ `...` (đánh dấu `# TODO`), chạy ô code, rồi chạy **ô tự kiểm tra** bên dưới (hiện ✅ nếu đúng).
4. Chỉ xem **lời giải** khi đã thử xong.

> Chạy ô *Chuẩn bị* ngay bên dưới trước tiên. Dữ liệu được sinh ngẫu nhiên có kiểm soát nên không cần tải gì.

## Bài toán
Một công ty bất động sản muốn ước lượng **giá nhà (triệu đồng)** dựa trên 4 đặc trưng: diện tích (m²), số phòng ngủ, tuổi nhà (năm) và khoảng cách đến trung tâm (km).

In [ ]:
# ===== CHUẨN BỊ (chạy ô này trước, không cần sửa) =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

rng = np.random.default_rng(2026)
n = 300
dien_tich = rng.uniform(30, 150, n)
so_phong = rng.integers(1, 6, n)
tuoi_nha = rng.uniform(0, 40, n)
khoang_cach = rng.uniform(1, 20, n)
gia = (500 + 30 * dien_tich + 120 * so_phong - 8 * tuoi_nha - 25 * khoang_cach
       + rng.normal(0, 150, n))

df = pd.DataFrame({"dien_tich": dien_tich, "so_phong": so_phong,
                   "tuoi_nha": tuoi_nha, "khoang_cach": khoang_cach, "gia": gia})
cot_dac_trung = ["dien_tich", "so_phong", "tuoi_nha", "khoang_cach"]
df.head()

## Bước 1. Khám phá dữ liệu

**Yêu cầu.** Tính ma trận tương quan của `df`, rồi cho biết đặc trưng nào tương quan mạnh nhất với `gia`. Lưu **tên cột** vào biến `dac_trung_manh_nhat`.

<details>
<summary>💡 Gợi ý 1</summary>

Pandas có sẵn phương thức `DataFrame.corr()` trả về ma trận tương quan.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Lấy cột tương quan với `gia`: `df.corr()['gia']`. Nhớ bỏ chính `gia` (tương quan = 1) ra.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Dùng `.drop('gia')`, rồi `.abs()` (chỉ quan tâm độ mạnh, không quan tâm dấu) và `.idxmax()` để lấy tên cột.

</details>

In [ ]:
tuong_quan = ...        # TODO: ma trận tương quan của df
dac_trung_manh_nhat = ...   # TODO: tên cột tương quan mạnh nhất với gia
print(tuong_quan["gia"].round(3))
print("Mạnh nhất:", dac_trung_manh_nhat)

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
tuong_quan = df.corr()
dac_trung_manh_nhat = tuong_quan["gia"].drop("gia").abs().idxmax()
print(tuong_quan["gia"].round(3))
print("Mạnh nhất:", dac_trung_manh_nhat)
```

</details>

In [ ]:
assert dac_trung_manh_nhat == "dien_tich", "Chưa đúng, thử lại nhé!"
print("✅ Bước 1 đúng!")

## Bước 2. Chia tập huấn luyện / kiểm tra

**Yêu cầu.** Chia dữ liệu: đặc trưng `X = df[cot_dac_trung]`, nhãn `y = df['gia']`; 80% huấn luyện, 20% kiểm tra, `random_state=42`. Đặt tên `X_train, X_test, y_train, y_test`.

<details>
<summary>💡 Gợi ý 1</summary>

Dùng `train_test_split` đã import sẵn.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Cú pháp: `train_test_split(X, y, test_size=..., random_state=...)` trả về **4** đối tượng theo thứ tự `X_train, X_test, y_train, y_test`.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

`test_size=0.2` nghĩa là 20% làm tập kiểm tra.

</details>

In [ ]:
X = ...   # TODO
y = ...   # TODO
X_train, X_test, y_train, y_test = ...   # TODO
print(X_train.shape, X_test.shape)

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
X = df[cot_dac_trung]
y = df["gia"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
```

</details>

In [ ]:
assert X_train.shape == (240, 4) and X_test.shape == (60, 4), "Kiểm tra lại test_size và X."
print("✅ Bước 2 đúng!")

## Bước 3. Hồi quy đơn biến theo diện tích

**Yêu cầu.** Huấn luyện `LinearRegression` chỉ với cột `dien_tich`. Tính `r2_don` (R² trên tập kiểm tra).

<details>
<summary>💡 Gợi ý 1</summary>

`X_train[['dien_tich']]` (hai cặp ngoặc) giữ dạng bảng 2 chiều — scikit-learn yêu cầu X là 2 chiều.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Quy trình 3 dòng: `mh = LinearRegression()` → `mh.fit(X_tr, y_train)` → `mh.predict(X_te)`.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

R² = `r2_score(y_test, y_du_doan)`. Nhớ **đánh giá trên tập kiểm tra**, không phải tập huấn luyện.

</details>

In [ ]:
mh_don = ...            # TODO: tạo và huấn luyện mô hình với cột dien_tich
y_dd_don = ...          # TODO: dự đoán trên X_test[['dien_tich']]
r2_don = ...            # TODO
print(f"Đơn biến: w0 = {mh_don.intercept_:.1f}, w1 = {mh_don.coef_[0]:.2f}, R² = {r2_don:.4f}")

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
mh_don = LinearRegression().fit(X_train[["dien_tich"]], y_train)
y_dd_don = mh_don.predict(X_test[["dien_tich"]])
r2_don = r2_score(y_test, y_dd_don)
print(f"Đơn biến: w0 = {mh_don.intercept_:.1f}, w1 = {mh_don.coef_[0]:.2f}, R² = {r2_don:.4f}")
```

</details>

In [ ]:
assert 0.5 < r2_don < 0.95, "R² có vẻ chưa hợp lý, kiểm tra lại dữ liệu đưa vào."
print("✅ Bước 3 đúng!")

## Bước 4. Hồi quy đa biến và diễn giải hệ số

**Yêu cầu.** Huấn luyện mô hình với **cả 4** đặc trưng. Tính `r2_da`, `rmse_da` trên tập kiểm tra và lưu hệ số vào Series `he_so` (chỉ số là tên cột). Hệ số nào âm? Ý nghĩa là gì?

<details>
<summary>💡 Gợi ý 1</summary>

Giống Bước 3 nhưng dùng toàn bộ `X_train`, `X_test`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

`rmse = np.sqrt(mean_squared_error(y_test, y_dd))`.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Tạo Series: `pd.Series(mh.coef_, index=cot_dac_trung)`.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

Gợi ý diễn giải: hệ số của `tuoi_nha` âm nghĩa là — giữ các biến khác cố định — nhà cũ thêm 1 năm thì giá giảm trung bình |w| triệu đồng.

</details>

In [ ]:
mh_da = ...        # TODO
y_dd_da = ...      # TODO
r2_da = ...        # TODO
rmse_da = ...      # TODO
he_so = ...        # TODO: pd.Series các hệ số
print(f"R² = {r2_da:.4f}, RMSE = {rmse_da:.1f}")
print(he_so.round(2))

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
mh_da = LinearRegression().fit(X_train, y_train)
y_dd_da = mh_da.predict(X_test)
r2_da = r2_score(y_test, y_dd_da)
rmse_da = np.sqrt(mean_squared_error(y_test, y_dd_da))
he_so = pd.Series(mh_da.coef_, index=cot_dac_trung)
print(f"R² = {r2_da:.4f}, RMSE = {rmse_da:.1f}")
print(he_so.round(2))
```

</details>

In [ ]:
assert r2_da > r2_don, "Mô hình đa biến nên tốt hơn đơn biến!"
assert he_so["tuoi_nha"] < 0 and he_so["khoang_cach"] < 0 and he_so["dien_tich"] > 0
print("✅ Bước 4 đúng! Hệ số học được gần với giá trị thật (30, 120, -8, -25).")

## Bước 5. Tự cài đặt OLS (phương trình chuẩn)

**Yêu cầu.** Tính `w_ols` = $(X^\top X)^{-1}X^\top y$ trên tập huấn luyện, **thêm cột 1** cho hệ số chặn. Kết quả phải trùng `mh_da`.

<details>
<summary>💡 Gợi ý 1</summary>

Thêm cột 1 vào đầu: `Xb = np.c_[np.ones(len(X_train)), X_train.values]`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Công thức: `w = (XᵀX)⁻¹ Xᵀ y`. Trong NumPy phép chuyển vị là `Xb.T`, nhân ma trận là `@`.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Dùng `np.linalg.solve(Xb.T @ Xb, Xb.T @ y_train)` — ổn định hơn `np.linalg.inv`.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

`w_ols[0]` là hệ số chặn, `w_ols[1:]` là hệ số các đặc trưng.

</details>

In [ ]:
Xb = ...       # TODO: thêm cột 1
w_ols = ...    # TODO
print("w_ols      :", np.round(w_ols, 3))
print("sklearn    :", np.round(np.r_[mh_da.intercept_, mh_da.coef_], 3))

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
Xb = np.c_[np.ones(len(X_train)), X_train.values]
w_ols = np.linalg.solve(Xb.T @ Xb, Xb.T @ y_train.values)
print("w_ols      :", np.round(w_ols, 3))
print("sklearn    :", np.round(np.r_[mh_da.intercept_, mh_da.coef_], 3))
```

</details>

In [ ]:
assert np.allclose(w_ols, np.r_[mh_da.intercept_, mh_da.coef_], atol=1e-6), "OLS chưa trùng sklearn."
print("✅ Bước 5 đúng!")

## Bước 6. Gradient Descent đa biến

**Yêu cầu.** Cài đặt `gd(X, y, alpha, so_lan)` cho nhiều đặc trưng. **Chuẩn hoá** đặc trưng trước (trừ trung bình, chia độ lệch chuẩn), rồi chạy GD và vẽ đường cong MSE. Với `alpha=0.1`, `so_lan=500`, kết quả phải đạt R² gần với OLS.

<details>
<summary>💡 Gợi ý 1</summary>

Vì sao chuẩn hoá? Các đặc trưng có thang rất khác nhau (diện tích ~100, số phòng ~3), GD trên dữ liệu chưa chuẩn hoá hội tụ rất chậm hoặc phân kỳ.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Chuẩn hoá: `mu = X_train.mean()`, `sd = X_train.std()`; `Xs = ((X_train - mu) / sd).values`. Dùng **mu, sd của tập huấn luyện** cho cả tập kiểm tra.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Mỗi vòng lặp: `y_hat = Xb @ w`; `loi = y - y_hat`; `grad = -2/n * Xb.T @ loi`; `w = w - alpha * grad`.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

Lưu `np.mean(loi**2)` mỗi vòng vào danh sách để vẽ đường cong mất mát. Khởi tạo `w = np.zeros(Xb.shape[1])`.

</details>

<details>
<summary>💡 Gợi ý 5</summary>

Để dự đoán trên tập kiểm tra: chuẩn hoá `X_test` bằng mu, sd cũ, thêm cột 1, rồi `Xb_te @ w`.

</details>

In [ ]:
mu = ...    # TODO
sd = ...    # TODO
Xs_tr = ((X_train - mu) / sd).values
Xs_te = ((X_test - mu) / sd).values

def gd(X, y, alpha=0.1, so_lan=500):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    lich_su = []
    for _ in range(so_lan):
        # TODO: tính y_hat, loi, grad, cập nhật w, lưu MSE vào lich_su
        ...
    return w, lich_su

w_gd, mat_mat = gd(Xs_tr, y_train.values)
y_dd_gd = np.c_[np.ones(len(Xs_te)), Xs_te] @ w_gd
r2_gd = r2_score(y_test, y_dd_gd)
print(f"R² (GD) = {r2_gd:.4f}   |   R² (OLS) = {r2_da:.4f}")

plt.plot(mat_mat); plt.yscale("log")
plt.xlabel("Vòng lặp"); plt.ylabel("MSE"); plt.title("Đường cong mất mát"); plt.show()

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
mu = X_train.mean()
sd = X_train.std()
Xs_tr = ((X_train - mu) / sd).values
Xs_te = ((X_test - mu) / sd).values

def gd(X, y, alpha=0.1, so_lan=500):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    lich_su = []
    n = len(y)
    for _ in range(so_lan):
        y_hat = Xb @ w
        loi = y - y_hat
        grad = -2 / n * Xb.T @ loi
        w = w - alpha * grad
        lich_su.append(np.mean(loi ** 2))
    return w, lich_su

w_gd, mat_mat = gd(Xs_tr, y_train.values)
y_dd_gd = np.c_[np.ones(len(Xs_te)), Xs_te] @ w_gd
r2_gd = r2_score(y_test, y_dd_gd)
print(f"R² (GD) = {r2_gd:.4f}   |   R² (OLS) = {r2_da:.4f}")

plt.plot(mat_mat); plt.yscale("log")
plt.xlabel("Vòng lặp"); plt.ylabel("MSE"); plt.title("Đường cong mất mát"); plt.show()
```

</details>

In [ ]:
assert mat_mat[-1] < mat_mat[0], "Mất mát phải giảm theo thời gian."
assert abs(r2_gd - r2_da) < 0.01, "R² của GD chưa gần OLS: kiểm tra grad và việc chuẩn hoá."
print("✅ Bước 6 đúng!")

## Bước 7. Phân tích phần dư

**Yêu cầu.** Tính phần dư trên tập kiểm tra của mô hình đa biến (`phan_du`) và vẽ 2 biểu đồ: phần dư theo giá trị dự đoán, và histogram phần dư. Cho nhận xét: các giả định có thoả mãn không?

<details>
<summary>💡 Gợi ý 1</summary>

Phần dư = giá trị thật − giá trị dự đoán: `y_test - y_dd_da`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Biểu đồ 1: `plt.scatter(y_dd_da, phan_du)` và `plt.axhline(0)`. Dấu hiệu tốt: điểm rải đều quanh 0, không có hình phễu hay đường cong.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Biểu đồ 2: `plt.hist(phan_du, bins=20)`. Dấu hiệu tốt: hình chuông quanh 0.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

Nhận xét kỳ vọng: dữ liệu được sinh đúng theo mô hình tuyến tính với nhiễu chuẩn, nên phần dư phải rải ngẫu nhiên và gần chuẩn.

</details>

In [ ]:
phan_du = ...   # TODO
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# TODO: vẽ scatter phần dư theo dự đoán ở ax[0], histogram ở ax[1]
plt.tight_layout(); plt.show()
print(f"Trung bình phần dư = {np.mean(phan_du):.2f}, độ lệch chuẩn = {np.std(phan_du):.2f}")

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
phan_du = y_test - y_dd_da
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(y_dd_da, phan_du, alpha=0.7)
ax[0].axhline(0, color="crimson", ls="--")
ax[0].set_xlabel("Giá dự đoán"); ax[0].set_ylabel("Phần dư")
ax[1].hist(phan_du, bins=20, color="steelblue", edgecolor="white")
ax[1].set_xlabel("Phần dư")
plt.tight_layout(); plt.show()
print(f"Trung bình phần dư = {np.mean(phan_du):.2f}, độ lệch chuẩn = {np.std(phan_du):.2f}")
```

</details>

In [ ]:
assert abs(np.mean(phan_du)) < 60, "Phần dư phải có trung bình gần 0."
print("✅ Bước 7 đúng!")

## Bước 8. Dự đoán nhà mới

**Yêu cầu.** Dự đoán giá của căn nhà: 80 m², 3 phòng ngủ, 10 năm tuổi, cách trung tâm 5 km. Lưu vào `gia_du_doan`. Sau đó tự tính tay bằng `he_so` và `mh_da.intercept_` để kiểm tra.

<details>
<summary>💡 Gợi ý 1</summary>

Mô hình cần đầu vào là bảng có **đúng thứ tự cột** `cot_dac_trung`: `pd.DataFrame([[80, 3, 10, 5]], columns=cot_dac_trung)`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

`mh_da.predict(...)` trả về mảng; lấy phần tử đầu `[0]`.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Tính tay: `mh_da.intercept_ + (he_so * np.array([80, 3, 10, 5])).sum()`.

</details>

In [ ]:
nha_moi = pd.DataFrame([[80, 3, 10, 5]], columns=cot_dac_trung)
gia_du_doan = ...    # TODO
gia_tinh_tay = ...   # TODO
print(f"Giá dự đoán: {gia_du_doan:.0f} triệu đồng (tính tay: {gia_tinh_tay:.0f})")

<details>
<summary>✅ Xem lời giải (chỉ mở sau khi đã tự làm!)</summary>

```python
nha_moi = pd.DataFrame([[80, 3, 10, 5]], columns=cot_dac_trung)
gia_du_doan = mh_da.predict(nha_moi)[0]
gia_tinh_tay = mh_da.intercept_ + (he_so * np.array([80, 3, 10, 5])).sum()
print(f"Giá dự đoán: {gia_du_doan:.0f} triệu đồng (tính tay: {gia_tinh_tay:.0f})")
```

</details>

In [ ]:
assert abs(gia_du_doan - gia_tinh_tay) < 1e-6
assert 2500 < gia_du_doan < 3500
print("✅ Bước 8 đúng!")

## Thử thách mở rộng (không có đáp án mẫu)
1. **Thiếu đặc trưng:** bỏ `dien_tich` khỏi mô hình đa biến. $R^2$ giảm bao nhiêu? Điều đó nói gì về tầm quan trọng của biến này?
2. **Đa cộng tuyến:** thêm cột `dien_tich_ft2 = dien_tich * 10.764` (cùng thông tin, khác đơn vị) rồi huấn luyện lại. Hệ số thay đổi thế nào? Giải thích vì sao ma trận $X^\top X$ gần suy biến.
3. **Ngoại lai:** đổi 3 giá trị `gia` thành $10\times$ giá ban đầu, huấn luyện lại và so sánh hệ số, $R^2$, MAE. Hồi quy tuyến tính (MSE) nhạy với ngoại lai ra sao?
4. **Tốc độ học:** chạy `gd` với `alpha` = 0.001, 0.1, 1.5. Vẽ ba đường cong mất mát trên cùng một hình.
5. **Phi tuyến:** đặt lại `gia` theo $\sqrt{\text{dien\_tich}}$, hồi quy tuyến tính, quan sát phần dư; sau đó thêm đặc trưng bậc 2 bằng `PolynomialFeatures`.

## Tổng kết
| Bước | Kỹ năng |
|---|---|
| 1 | Khám phá, tương quan |
| 2 | Chia train/test |
| 3–4 | Mô hình đơn biến, đa biến, diễn giải hệ số |
| 5 | Phương trình chuẩn (OLS) |
| 6 | Gradient Descent + chuẩn hoá |
| 7 | Chẩn đoán phần dư |
| 8 | Dự đoán mẫu mới |